In [1]:
from whar_datasets import (
    Loader,
    LOSOSplitter,
    PostProcessingPipeline,
    PreProcessingPipeline,
    TorchAdapter,
    WHARDatasetID,
    get_dataset_cfg,
)

In [2]:
# create cfg for WISDM dataset
cfg = get_dataset_cfg(WHARDatasetID.WISDM, "../datasets")
cfg.parallelize = True

# create and run pre-processing pipeline
pre_pipeline = PreProcessingPipeline(cfg)
activity_df, session_df, window_df = pre_pipeline.run()

# create LOSO splits
splitter = LOSOSplitter(cfg)
splits = splitter.get_splits(session_df, window_df)
split = splits[3]

# create and run post-processing pipeline for the specific split
post_pipeline = PostProcessingPipeline(
    cfg, pre_pipeline, window_df, split.train_indices
)
samples = post_pipeline.run()

# create dataloaders for the specific split
loader = Loader(session_df, window_df, post_pipeline.samples_dir, samples)
adapter = TorchAdapter(cfg, loader, split)
dataloaders = adapter.get_dataloaders(batch_size=64)
print(f"num subjects / splits: {cfg.num_of_subjects}/{len(splits)}")
print(f"num channels: {len(cfg.sensor_channels)}")

2026-02-04 10:36:37,709 - whar-datasets - INFO - Running DownloadingStep
2026-02-04 10:36:37,710 - whar-datasets - INFO - Checking hash for DownloadingStep
2026-02-04 10:36:37,711 - whar-datasets - INFO - Hash is up to date
2026-02-04 10:36:37,711 - whar-datasets - INFO - Running ParsingStep
2026-02-04 10:36:37,712 - whar-datasets - INFO - Checking hash for ParsingStep
2026-02-04 10:36:37,713 - whar-datasets - INFO - Hash is up to date
2026-02-04 10:36:37,713 - whar-datasets - INFO - Running WindowingStep
2026-02-04 10:36:37,713 - whar-datasets - INFO - Checking hash for WindowingStep
2026-02-04 10:36:37,714 - whar-datasets - INFO - Hash is up to date
2026-02-04 10:36:37,714 - whar-datasets - INFO - Loading windowing
2026-02-04 10:36:37,725 - whar-datasets - INFO - activity_ids from 0 to 5
2026-02-04 10:36:37,726 - whar-datasets - INFO - subject_ids from 0 to 35
2026-02-04 10:36:37,827 - whar-datasets - INFO - Running SamplingStep
2026-02-04 10:36:37,827 - whar-datasets - INFO - Checki

num subjects / splits: 36/36
num channels: 3


In [3]:
train_loader, val_loader, test_loader = (
    dataloaders["train"],
    dataloaders["val"],
    dataloaders["test"],
)

print(f"Number of training samples: {len(train_loader)}")
print(f"Number of validation samples: {len(val_loader)}")
print(f"Number of test samples: {len(test_loader)}")

print(len(split.train_indices))
print(len(split.val_indices))
print(len(split.test_indices))

ya, ys, sample = loader.sample_items(1)

print(ya[0])
print(ys[0])
print(sample[0][0].shape)

Number of training samples: 271
Number of validation samples: 12
Number of test samples: 68
17336
4333
739
0
4
(100, 3)


In [4]:
import torch
from meta_learning.training.trainer import Trainer
from meta_learning.models.tiny_har import TinyHAR

model = TinyHAR(
    input_channels=len(cfg.sensor_channels),
    window_size=int(cfg.window_time * cfg.sampling_freq),
    num_classes=cfg.num_of_activities,
    # num_filters=64,
)


optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss()
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")


trainer = Trainer(
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
)

state_dict = trainer.fit(epochs=20, patience=5)

Starting training on mps for 20 epochs.

Epoch 1/20


Training:   0%|          | 0/271 [00:00<?, ?it/s]

Validation:   0%|          | 0/12 [00:00<?, ?it/s]

Testing:   0%|          | 0/68 [00:00<?, ?it/s]

Train: Loss 0.5334 | Acc 0.8283 | F1 0.7411
Val:   Loss 0.3239 | Acc 0.8999 | F1 0.8372
Test:  Loss 0.2823 | Acc 0.9068 | F1 0.8547

Epoch 2/20


...

Train: Loss 0.0968 | Acc 0.9682 | F1 0.9545
Val:   Loss 0.4435 | Acc 0.8985 | F1 0.8524
Test:  Loss 0.1235 | Acc 0.9615 | F1 0.9487

Epoch 7/20


Training:   0%|          | 0/271 [00:00<?, ?it/s]

Validation:   0%|          | 0/12 [00:00<?, ?it/s]

Testing:   0%|          | 0/68 [00:00<?, ?it/s]

Train: Loss 0.0843 | Acc 0.9731 | F1 0.9622
Val:   Loss 0.2553 | Acc 0.9215 | F1 0.8906
Test:  Loss 0.1031 | Acc 0.9668 | F1 0.9515
Early stopping triggered after 7 epochs.
Restoring best model weights...
